In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA

from data_loaders.mice import MiceDatasetLoader

## test data loader for mice model

In [ ]:
# uncoupled no baiting
# subject_ids = [774212]
# subject_ids = [793446]

# uncoupled baiting
subject_ids = [790724]

# mixed
# subject_ids = [774212, 790724]


loader = MiceDatasetLoader(
    subject_ids=subject_ids,
    ignore_policy='exclude',
    features={'animal_response': 'prev choice', 'rewarded': 'prev reward'},
    eval_every_n=2,
    multisubject=False,
    seed=42,
)
dataset_bundle = loader.load()

In [ ]:
dataset_bundle

## plot latents from trained dis-rnn

In [ ]:
from disentangled_rnns.library import disrnn, plotting, rnn_utils

In [ ]:
fig_dpi = 300

def plot_traj_in_PC_space(
    transformed_RNN_hidden_states_background,
    background_coloring_array,
    background_colorbar_label,
    background_coloring_minmax,
    num_pcs2plot,
    transformed_RNN_hidden_states_example_run=None,
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
):
    """
    Plot the transformed hidden states in the PCA space.
    Args:
        transformed_RNN_hidden_states_background: Transformed RNN hidden states for background
        transformed_RNN_hidden_states_example_run: Transformed RNN hidden states for an example run
        coloring_array: Array to color each state
        num_pcs2plot: Number of principal components to plot
        
    Returns:
        fig: matplotlib figure
    """
    n_axs = num_pcs2plot * (num_pcs2plot-1) / 2
    n_cols = num_pcs2plot
    n_rows = int(n_axs/ n_cols) + (n_axs% n_cols > 0)
    
    if n_rows == 1:
        n_rows = 2

    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(6*n_cols, 4*n_rows), dpi=fig_dpi
    )

    ax_id = 0
    for pc_x in range(num_pcs2plot):
        for pc_y in range(num_pcs2plot):
            if pc_x < pc_y:
                ax = axs[int(ax_id/n_cols), ax_id%n_cols]
                
                # set the title
                if transformed_fixed_points is not None:
                    ax.set_title(f'Fixed points under {fixed_points_condition}', fontsize=18)
                else:
                    ax.set_title(f'Neural space trajectoy', fontsize=18)
                
                # Plot the background
                scatter = ax.scatter(
                    transformed_RNN_hidden_states_background[:, :, pc_x],
                    transformed_RNN_hidden_states_background[:, :, pc_y],
                    s=2.5,
                    c=background_coloring_array, cmap=cm.coolwarm,
                    vmin=background_coloring_minmax[0], 
                    vmax=background_coloring_minmax[1],
                    alpha=0.9
                )
                ax.set_xlabel(f'PC {pc_x}', fontsize=18)
                ax.set_ylabel(f'PC {pc_y}', fontsize=18)
                ax.tick_params(axis='both', which='major', labelsize=12)
                fig.colorbar(
                    scatter, ax=ax,
                    label=background_colorbar_label
                )
                
                # Plot the trajectory of the example run
                if transformed_RNN_hidden_states_example_run is not None:
                    ax.plot(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        color='k', lw=1, alpha=0.9
                    )
                    exameple_run_len = transformed_RNN_hidden_states_example_run.shape[0]
                    example_run = ax.scatter(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        s=4,
                        c=np.arange(exameple_run_len), cmap=cm.copper,
                        vmin=0, vmax=exameple_run_len
                    )  # color coded by time step
                    fig.colorbar(
                        example_run, ax=ax,
                        label='Time step'
                    )

                # Plot the fixed points
                if transformed_fixed_points is not None:
                    scatter_fp = ax.scatter(
                        transformed_fixed_points[:, pc_x],
                        transformed_fixed_points[:, pc_y],
                        marker='o', s=5,
                        c=fixed_points_coloring_array, cmap=cm.gray,
                        vmin=1e-13, vmax=10, norm='log',
                        alpha=0.6
                    )  # color coded by q values
                    fig.colorbar(
                        scatter_fp, ax=ax,
                        label='Kinetic Energy (q value)'
                    )

                ax_id += 1

    fig.tight_layout()
    return fig


def plot_traj_in_PC_space_by_outcome(
    transformed_RNN_hidden_states,
    prev_choices,
    prev_rewards,
    num_pcs2plot,
    transformed_RNN_hidden_states_example_run=None,
    example_run_choices=None,
    example_run_rewards=None
):
    """
    Plot the transformed hidden states in PCA space, colored by previous outcome.
    
    Args:
        transformed_RNN_hidden_states: Transformed RNN hidden states [n_trials, n_sessions, n_pcs]
        prev_choices: Previous choices [n_trials, n_sessions], 0=left, 1=right
        prev_rewards: Previous rewards [n_trials, n_sessions], 0=unrewarded, 1=rewarded
        num_pcs2plot: Number of principal components to plot
        transformed_RNN_hidden_states_example_run: Optional example trajectory
        example_run_choices: Choices for example run
        example_run_rewards: Rewards for example run
        
    Returns:
        fig: matplotlib figure
    """
    # Create outcome categories: 
    # 0=rewarded_left, 1=unrewarded_left, 2=rewarded_right, 3=unrewarded_right
    outcome_categories = np.zeros_like(prev_choices)
    outcome_categories[(prev_choices == 0) & (prev_rewards == 1)] = 0  # rewarded left
    outcome_categories[(prev_choices == 0) & (prev_rewards == 0)] = 1  # unrewarded left
    outcome_categories[(prev_choices == 1) & (prev_rewards == 1)] = 2  # rewarded right
    outcome_categories[(prev_choices == 1) & (prev_rewards == 0)] = 3  # unrewarded right
    
    # Print verification statistics
    print("Outcome category counts:")
    print(f"  Rewarded left (0): {np.sum(outcome_categories == 0)}")
    print(f"  Unrewarded left (1): {np.sum(outcome_categories == 1)}")
    print(f"  Rewarded right (2): {np.sum(outcome_categories == 2)}")
    print(f"  Unrewarded right (3): {np.sum(outcome_categories == 3)}")
    
    # Define distinct colors for each outcome
    colors = {
        0: 'green',      # rewarded left
        1: 'lightcoral', # unrewarded left
        2: 'blue',       # rewarded right
        3: 'orange'      # unrewarded right
    }
    labels = {
        0: 'Rewarded Left',
        1: 'Unrewarded Left',
        2: 'Rewarded Right',
        3: 'Unrewarded Right'
    }
    
    n_axs = num_pcs2plot * (num_pcs2plot-1) / 2
    n_cols = num_pcs2plot
    n_rows = int(n_axs / n_cols) + (n_axs % n_cols > 0)
    
    if n_rows == 1:
        n_rows = 2

    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(5*n_cols, 5*n_rows), dpi=fig_dpi
    )

    ax_id = 0
    for pc_x in range(num_pcs2plot):
        for pc_y in range(num_pcs2plot):
            if pc_x < pc_y:
                ax = axs[int(ax_id/n_cols), ax_id%n_cols]
                
                ax.set_title(f'Colored by previous outcome', fontsize=18)
                
                # Plot each outcome category separately
                for cat in range(4):
                    mask = (outcome_categories == cat)
                    if np.any(mask):
                        ax.scatter(
                            transformed_RNN_hidden_states[:, :, pc_x][mask],
                            transformed_RNN_hidden_states[:, :, pc_y][mask],
                            s=2.5,
                            c=colors[cat],
                            label=labels[cat],
                            alpha=0.6
                        )
                
                ax.set_xlabel(f'PC {pc_x}', fontsize=18)
                ax.set_ylabel(f'PC {pc_y}', fontsize=18)
                ax.tick_params(axis='both', which='major', labelsize=12)
                
                # Add legend only to first subplot
                if ax_id == 0:
                    ax.legend(fontsize=12, markerscale=4, loc='best')
                
                # Plot example trajectory if provided
                if transformed_RNN_hidden_states_example_run is not None:
                    # Plot trajectory line
                    ax.plot(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        color='k', lw=2, alpha=0.5, zorder=100
                    )
                    
                    # Plot example run points colored by outcome
                    if example_run_choices is not None and example_run_rewards is not None:
                        example_categories = np.zeros_like(example_run_choices)
                        example_categories[(example_run_choices == 0) & (example_run_rewards == 1)] = 0
                        example_categories[(example_run_choices == 0) & (example_run_rewards == 0)] = 1
                        example_categories[(example_run_choices == 1) & (example_run_rewards == 1)] = 2
                        example_categories[(example_run_choices == 1) & (example_run_rewards == 0)] = 3
                        
                        for cat in range(4):
                            mask = (example_categories == cat)
                            if np.any(mask):
                                ax.scatter(
                                    transformed_RNN_hidden_states_example_run[:, pc_x][mask],
                                    transformed_RNN_hidden_states_example_run[:, pc_y][mask],
                                    s=20,
                                    c=colors[cat],
                                    edgecolors='black',
                                    linewidths=0.5,
                                    alpha=0.9,
                                    zorder=101
                                )

                ax_id += 1

    fig.tight_layout()
    return fig


# Verification - test the function
def verify_outcome_categorization(choices, rewards, n_trials_to_check=10):
    """Helper function to verify outcome categorization is correct."""
    print("\nVerification of first few trials:")
    print("Trial | Choice | Reward | Expected Category | Actual Category")
    print("-" * 70)
    
    for i in range(min(n_trials_to_check, len(choices))):
        choice = choices[i]
        reward = rewards[i]
        
        # Expected category
        if choice == 0 and reward == 1:
            expected = "Rewarded Left (0)"
            expected_val = 0
        elif choice == 0 and reward == 0:
            expected = "Unrewarded Left (1)"
            expected_val = 1
        elif choice == 1 and reward == 1:
            expected = "Rewarded Right (2)"
            expected_val = 2
        elif choice == 1 and reward == 0:
            expected = "Unrewarded Right (3)"
            expected_val = 3
        else:
            expected = "Invalid"
            expected_val = -1
        
        # Actual category
        outcome_cat = np.zeros(1, dtype=int)
        if choice == 0 and reward == 1:
            outcome_cat[0] = 0
        elif choice == 0 and reward == 0:
            outcome_cat[0] = 1
        elif choice == 1 and reward == 1:
            outcome_cat[0] = 2
        elif choice == 1 and reward == 0:
            outcome_cat[0] = 3
            
        match = "✓" if outcome_cat[0] == expected_val else "✗"
        print(f"{i:5d} | {choice:6.0f} | {reward:6.0f} | {expected:25s} | {outcome_cat[0]} {match}")

In [ ]:
import json

# uncoupled no baiting
# trained_model_fp = 

# uncoupled baiting
trained_model_fp = '/root/capsule/data/results-mice-uncoupled_baiting-790724-051437f2-9c43-4576-94ea-e9a524671b86'
# model_id = '9', '10'

# combined
# trained_model_fp = '/root/capsule/data/results-mice-uncoupled_no_baiting-774212-ef48dbe5-0269-45fd-ad01-2bd0fb9d568a'


model_id = '9'


with open(
    f'{trained_model_fp}/{model_id}/inputs/jobs/.hydra/config.json', 
    'r'
) as fp:
    disrnn_config_json = json.load(fp)

with open(
    f'{trained_model_fp}/{model_id}/outputs/params.json', 
    'r'
) as fp:
    params = rnn_utils.to_np(json.load(fp))


# params_disrnn = params
params_disrnn = params['hk_disentangled_rnn']

In [ ]:
for key in params.keys():
    print(key)

In [ ]:
dataset_eval = dataset_bundle.eval_set

disrnn_config = disrnn.DisRnnConfig(
    # Dataset related
    obs_size=2,  # Choice, reward
    output_size=2,  # Choose left / choose right
    x_names=dataset_eval.x_names,
    y_names=dataset_eval.y_names,
    # Network architecture
    latent_size=disrnn_config_json['model']['architecture']['latent_size'],
    update_net_n_units_per_layer=disrnn_config_json['model']['architecture']['update_net_n_units_per_layer'],
    update_net_n_layers=disrnn_config_json['model']['architecture']['update_net_n_layers'],
    choice_net_n_units_per_layer=disrnn_config_json['model']['architecture']['choice_net_n_units_per_layer'],
    choice_net_n_layers=disrnn_config_json['model']['architecture']['choice_net_n_layers'],
    activation=disrnn_config_json['model']['architecture']['activation'],
    # Penalties
    # noiseless_mode=False,
    noiseless_mode=True,
    latent_penalty=disrnn_config_json['model']['penalties']['latent_penalty'],
    choice_net_latent_penalty=disrnn_config_json['model']['penalties']['choice_net_latent_penalty'],
    update_net_obs_penalty=disrnn_config_json['model']['penalties']['update_net_obs_penalty'],
    update_net_latent_penalty=disrnn_config_json['model']['penalties']['update_net_latent_penalty'],
    l2_scale=0,
)

In [ ]:
xs, ys = dataset_eval.get_all()
print(f'xs shape: {xs.shape}, ys shape: {ys.shape}')

network_output, network_states = rnn_utils.eval_network(
    lambda: disrnn.HkDisentangledRNN(disrnn_config), params, xs)

# Compute normalized likelihood
logits = network_output[:,:,:2]  # First n_actions elements of network output are the logits (the final one is the penalty)
normalized_likelihood = rnn_utils.normalized_likelihood(labels = ys, output_logits=logits)
print(f'Normalized likelihood: {100*normalized_likelihood:.2f}%')

In [ ]:
# Plot network activations on an example session
example_session = 0

choices = xs[:, example_session, 0]
rewards = xs[:, example_session, 1]
scalars = network_states[:, example_session, :]

print(f'Choices: {choices}')
print(f'Rewards: {rewards}')

# two_armed_bandits.plot_2ab_sessdata(choices,
#                                     rewards,
#                                     scalars=scalars,
#                                     scalar_types='agent_states',
#                                     show_legend=False)
# plt.plot(scalars)

In [ ]:
# plot choice/reward history and latents
open_latents = [0,4]


# Filter out padding (where choices or rewards are -1)
valid_mask = (choices != -1) & (rewards != -1)
valid_indices = np.where(valid_mask)[0]
choices_filtered = choices[valid_mask]
rewards_filtered = rewards[valid_mask]
scalars_filtered = scalars[valid_mask]

fig, axs = plt.subplots(
    nrows=3, ncols=1, 
    figsize=(12, 10), 
    dpi=fig_dpi, sharex=True
)

# Top subplot: choice/reward history
ax = axs[0]
choice_mask = (choices_filtered == 0) | (choices_filtered == 1)
non_choice_mask = ~choice_mask
rewarded_mask = (rewards_filtered == 1) & choice_mask
unrewarded_mask = (rewards_filtered == 0) & choice_mask

ax.scatter(
    valid_indices[non_choice_mask],
    choices_filtered[non_choice_mask],
    marker='|',
    s=800,
    color='gray',
    label='No choice',
    alpha=0.7
)
ax.scatter(
    valid_indices[unrewarded_mask],
    choices_filtered[unrewarded_mask],
    marker='|',
    s=400,
    color='black',
    label='Unrewarded',
    alpha=0.7
)
ax.scatter(
    valid_indices[rewarded_mask],
    choices_filtered[rewarded_mask],
    marker='|',
    s=800,
    color='black',
    label='Rewarded',
    alpha=0.7
)
ax.set_ylim(-0.2, 1.2)
ax.set_yticks([0,1])
ax.set_yticklabels(['Left', 'Right'])
ax.set_title('Choices history', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

# middle subplot: latent values
ax = axs[1]
for latent_idx in open_latents:
    ax.plot(valid_indices, scalars_filtered[:, latent_idx], label=f'latent {latent_idx}')
ax.set_title('Open latents', fontsize=18)
ax.set_xlabel('Trial', fontsize=18)
ax.set_ylabel('Latent value', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

# bottom subplot: all latents
ax = axs[2]
all_latents = [0,1,2,3,4]
for latent_idx in all_latents:
    ax.plot(valid_indices, scalars_filtered[:, latent_idx], label=f'latent {latent_idx}')
ax.set_title('All latents', fontsize=18)
ax.set_xlabel('Trial', fontsize=18)
ax.set_ylabel('Latent value', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize=12)

fig.tight_layout()

In [ ]:
logit_sum = np.sum(np.exp(logits), axis=2)
action_prob = np.exp(logits)/ np.stack((logit_sum, logit_sum), axis=2)

In [ ]:
n_sessions = network_states.shape[1]

fig = plot_traj_in_PC_space(
    network_states[:,:,open_latents],
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=len(open_latents),
    transformed_RNN_hidden_states_example_run=network_states[:30,example_session,open_latents],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)

In [ ]:
# Extract previous choices and rewards from the input data
prev_choices = xs[:, :, 0]  # Shape: [n_trials, n_sessions]
prev_rewards = xs[:, :, 1]  # Shape: [n_trials, n_sessions]

# Optional: verify the outcome categorization on one session
verify_outcome_categorization(
    choices=prev_choices[:, 0],
    rewards=prev_rewards[:, 0],
    n_trials_to_check=10
)

# Plot trajectories colored by outcome
fig = plot_traj_in_PC_space_by_outcome(
    transformed_RNN_hidden_states=network_states[:,:,open_latents],
    prev_choices=prev_choices,
    prev_rewards=prev_rewards,
    num_pcs2plot=len(open_latents),
    transformed_RNN_hidden_states_example_run=network_states[:30, example_session, open_latents],
    example_run_choices=prev_choices[:30, example_session],
    example_run_rewards=prev_rewards[:30, example_session]
)

In [ ]:
# run pca on scalars and plot the first two pcs

# PCA using training set
pca_model = PCA(n_components=5)
pca_model.fit(network_states.reshape(-1, 5))
print(f'fitted disrnn, PCA explained variance (training): {pca_model.explained_variance_ratio_}')

# plot
fig, axs = plt.subplots(
    nrows=1, ncols=2,
    figsize=(6, 3), dpi=300
)

ax = axs[0]
ax.plot(pca_model.explained_variance_ratio_)
ax.set_title('fitted disrnn')
ax.set_xlabel('PC')
ax.set_ylabel('var. exp.')

ax = axs[1]
ax.plot(np.cumsum(pca_model.explained_variance_ratio_))
ax.set_title('cumulative var. explained')
ax.set_xlabel('PC')
ax.set_ylabel('cumulative var. exp.')
ax.set_ylim(0, 1)

fig.tight_layout()


In [ ]:
# PCA transformed training set
n_sessions = network_states.shape[1]
transformed_network_states = pca_model.transform(
    network_states.reshape(-1, 5)).\
    reshape(-1, n_sessions, 5)


# plt.scatter(
#     transformed_network_states[:,:,0],
#     transformed_network_states[:,:,1],
#     marker='o', s=5, alpha=0.85
# )
# plt.plot(
#     transformed_network_states[:160,0,0],
#     transformed_network_states[:160,0,1],
#     color='k', lw=1
# )


fig = plot_traj_in_PC_space(
    transformed_network_states,
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=5,
    transformed_RNN_hidden_states_example_run=transformed_network_states[:160,5,:],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)